# 住所→座標変換（address_geocode.ipynb）

## 基本の使い方

1. templates/residents.csv を元に住民CSVを作成
2. resident_id と address を入力
3. address_geocode.ipynb で座標変換
4. unmatched行の住所を修正
5. 同じCSVを再度 address_geocode.ipynb へ投入
6. 必要な行の変換が終わったら、そのCSVをそのまま sheltermatch.ipynb へ投入

住民CSVは常に `resident_id,address,latitude,longitude,geocode_status` の5列です。変換前後で列構成が
変わらないため、住所を修正して同じCSVを再度このNotebookへ投入（再変換。古い座標・状態は最新の結果へ
置き換わります）したり、変換が完了したCSVをそのまま `sheltermatch.ipynb` へアップロードしたりできます
（加工は不要です）。

このNotebookは、**共通住民CSVへ、公式アドレス・ベース・レジストリ（ABR）データから緯度経度を付与し、
sheltermatchでそのまま利用できるCSVを出力すること**だけを目的とした、独立した住所→座標変換ツールです。

- **sheltermatch本体（`sheltermatch.ipynb`）ではありません。** 避難所候補の算出・ハザード判定は行いません。
- 住所変換のロジック自体は `experiments/abr_csv_geocode_poc.ipynb`（PoC）で検証済みのものと同じです。
  PoCは検証履歴としてそのまま残しています。
- **ABRデータの取得方法は [docs/address-data.md](./docs/address-data.md) を参照してください。**

## 変換できる住所

- **地番住所**（例: `糸満市字○○673番地2`）: 町字＋地番（1〜3段）の完全一致で座標を付けます
- **住居表示住所**（町字マスタで `rsdt_addr_flg=1` の町字。例: `糸満市○○町37番14号`）: 町字＋街区符号＋
  住居番号（＋住居番号2）が住居表示データの1件に完全一致した場合だけ、その住居の位置参照の座標を付けます。
  街区だけが一致しても座標は付けません（街区の代表点や近い番号では代用しません）。
  `36番7-201号 ○○コーポ` のように空白区切りの方書が付いた3段の番号は、方書を外したうえでまず3段のまま
  完全一致を試し、一致しない場合だけ、3段目を除いた住居（`36番7号`）がABRに1件だけあり座標もあるときに、
  3段目を部屋番号等として除外して照合します。方書の無い `36番7-201号` は、3段目が部屋番号か確定できない
  ため、3段の完全一致が無ければ未変換のままです
- **方書付きの住所**（例: `…673番地2 ○○ホーム 101号室`）: 住所全文で見つからない場合だけ、元の住所の
  **空白の位置**で後ろの方書を外して照合し直します。`address` 列は書き換えません（出力CSVにも入力のまま
  残ります）。空白の無い地番住所（例: `673-2-101`）や、番号の後ろに空白を挟んで数字だけが続く住所
  （例: `673-2 101`。番号の続きか部屋番号か判断できない）は切り詰めません

曖昧な住所を推測して `matched` 扱いにすることはありません。

## 必要なABR ZIP（利用者が手動アップロード）

**以下のファイル名・コードは、既定の設定 `itoman-city`（沖縄県糸満市）での例です。** 別の自治体で使うときは、
`CONFIG_NAME` に対応する自治体の設定ファイル（`configs/<CONFIG_NAME>.json`）を用意し、その自治体の地番マスタ
（`mt_parcel_city<6桁の自治体コード>.csv.zip` 等）と、その都道府県単位の住居表示データを使います。

地番住所用（必須）:

1. `mt_town_all.csv.zip`（全国町字マスタ）
2. `mt_parcel_city472107.csv.zip`（糸満市 地番マスタ）
3. `mt_parcel_pos_city472107.csv.zip`（糸満市 地番マスタ位置参照拡張）

住居表示住所用（住居表示地域の住所を変換する場合に必要。**都道府県単位**の配布データ。例は沖縄県）:

4. `mt_rsdtdsp_blk_pref47.csv.zip`（住居表示-街区マスター）
5. `mt_rsdtdsp_blk_pos_pref47.csv.zip`（住居表示-街区マスター位置参照拡張。確認用で、座標には使いません）
6. `mt_rsdtdsp_rsdt_pref47.csv.zip`（住居表示-住居マスター）
7. `mt_rsdtdsp_rsdt_pos_pref47.csv.zip`（住居表示-住居マスター位置参照拡張）

住居表示データは都道府県単位（既定では沖縄県）で配布されているため、Notebook側でCSV内の `lg_code` を見て、
設定した自治体（既定では糸満市。`lg_code=472107`）の行だけを抽出して使います。住居表示データをアップロードしない場合、住居表示地域の
住所は `residential_display_area`（照合せず保留）になります。

ファイル名ではなく、ZIP内CSVのヘッダー列から種別を自動判定します。ABRデータの自動ダウンロードは行いません。
外部（GitHub）へ通信するのは、「設定」セルが自治体の設定ファイル（`configs/<CONFIG_NAME>.json`）とその読込モジュールを
取得するときだけです。住民CSV・住所・座標は、どこへも送信しません。

## Notebookの操作（セル実行順）

1. Notebookを開く
2. 「設定」セルを実行する（糸満市の通常利用では変更不要です。自治体名・自治体コード等は、設定ファイル
   `configs/itoman-city.json` から読み込みます。別の自治体で使うときだけ、`CONFIG_NAME` を変更します。
   設定ファイルを取得できない場合は、誤った設定で進めないよう、原因を表示して止まります）
3. 「ABRマスター準備」セルを実行し、ABR ZIP（地番用3種類＋住居表示用4種類）をまとめてアップロードする
4. 「住所変換ロジック定義」セルを実行する
5. 「住所CSV変換・出力」セルを実行し、共通住民CSV（`resident_id,address,latitude,longitude,geocode_status`）
   をアップロードする → 変換結果を確認し、`○○_geocoded.csv`をダウンロードする
6. 住所を修正した場合は、**ABR ZIPは再アップロードせず、「住所CSV変換・出力」セルだけを再実行**すれば、
   新しい変換結果CSVを取得できます。同じColabランタイムが生きている間は、住所CSVを差し替えるたびに
   ABRマスターを作り直す必要はありません。

## 共通住民CSVの形式

`resident_id,address,latitude,longitude,geocode_status` の5列が正式フォーマットです（テンプレート:
[templates/residents.csv](./templates/residents.csv)）。

| 列名 | 説明 |
| --- | --- |
| `resident_id` | 住民を一意に識別するID。空欄・重複不可（このNotebookが行を示して処理を止めます） |
| `address` | 変換対象の住所。空欄の行は `blank_address` として処理されます。入力のまま出力します |
| `latitude` | 変換前は空欄。変換後は緯度が入ります（再変換のたびに最新の値で上書きされます） |
| `longitude` | 変換前は空欄。変換後は経度が入ります（再変換のたびに最新の値で上書きされます） |
| `geocode_status` | 変換前は空欄。変換後は変換結果の状態が入ります（詳細は docs/address-data.md） |

出力CSVも同じ5列のみです。`normalized_address`・除外した方書（`ignored_suffix`）等のABR内部確認用の列は
Notebook上の確認表示にのみ使用し、CSVには含めません。方書を除外して照合できた住所の件数と一覧も、
Notebook上で確認できます。

## 個人情報の取り扱い（重要）

実際の住所データ・出力CSVには個人情報が含まれ得ます。**利用を許可された環境でのみ扱ってください。**
このリポジトリへ実際の個人情報をコミットしないでください。


In [ ]:
# ===== 設定 =====
# 自治体ごとの設定（自治体名・自治体コード・都道府県名）は、設定ファイル configs/<CONFIG_NAME>.json にまとめています
# （糸満市は "itoman-city"）。通常は変更不要です。別の自治体で使うときだけ、configs/ にあるその自治体の設定名へ
# 変更してください。このNotebookの住所変換のルール・出力の列（5列）は、設定ファイルでは変わりません。
CONFIG_NAME = "itoman-city"

# 以下は、このNotebookの既定値です。設定ファイルに同じ項目があるときは、設定ファイルの値が使われます
# （実行時に、最終的な値と設定元を表示します）。設定ファイルに項目が無いときだけ、ここの値を使います。
# 自治体名・自治体コードは、設定ファイルの必須項目のため、常に設定ファイルの値を使います。
PREF_NAME = "沖縄県"

import importlib.util
from pathlib import Path

import requests

# 設定ファイルと、その読込・検証を行うモジュール（src/config/municipality_config.py）は、GitHubの同じ版（タグ）から
# 取得します（sheltermatch.ipynb の外部モジュールと同じ版。取得先の組み立てはここの1か所だけです）。
# 同じ名前の設定ファイルが、このNotebookのあるフォルダ（またはその上位）の configs/ にあれば、それを使います。
SHELTERMATCH_CODE_REF = "v1.2.0"
GITHUB_RAW_REPO_URL = f"https://raw.githubusercontent.com/YanTKYS/sheltermatch/{SHELTERMATCH_CODE_REF}"
CONFIG_LOADER_API_VERSION = 1  # このNotebookが想定する、読込モジュールのAPIバージョン

config_module_url = f"{GITHUB_RAW_REPO_URL}/src/config/municipality_config.py"
try:
    response = requests.get(config_module_url, timeout=30)
    response.raise_for_status()
except Exception as error:
    raise RuntimeError(
        "設定ファイルの読込モジュールをGitHubから取得できませんでした。\n"
        "インターネット接続を確認して再実行してください。\n"
        f"（取得対象の版: {SHELTERMATCH_CODE_REF} / 取得先: {config_module_url} / 詳細: {error}）"
    ) from error
config_module_path = Path("sheltermatch_modules") / "municipality_config.py"
config_module_path.parent.mkdir(parents=True, exist_ok=True)
config_module_path.write_bytes(response.content)
spec = importlib.util.spec_from_file_location("municipality_config", config_module_path)
municipality_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(municipality_config)
if getattr(municipality_config, "CONFIG_LOADER_API_VERSION", None) != CONFIG_LOADER_API_VERSION:
    raise RuntimeError(
        "municipality_config の互換性を確認できません。\n"
        f"このNotebookが想定しているバージョン: {CONFIG_LOADER_API_VERSION} / "
        f"取得したモジュールのバージョン: {getattr(municipality_config, 'CONFIG_LOADER_API_VERSION', None)}"
        f"（版: {SHELTERMATCH_CODE_REF}）"
    )

# 設定ファイルを取得・解釈できないとき、必須項目が無いとき、型や値が不正なときは、Notebookの既定値で続行せず止まります。
config, config_origin = municipality_config.load_config(CONFIG_NAME, GITHUB_RAW_REPO_URL)

CITY_NAME = config["municipality"]["name"]
LG_CODE = municipality_config.local_government_code(config["municipality"]["code"])  # 検査数字つきの6桁（例: 472107）
PREF_NAME, pref_name_source = municipality_config.resolve_setting(config, "municipality", "prefecture", PREF_NAME)

print("設定を読み込みました。")
print("\n".join(municipality_config.describe_municipality(config, CONFIG_NAME, config_origin)))
print()
print("設定:")
print("\n".join(municipality_config.describe_settings([
    ("LG_CODE", LG_CODE, "JSON（自治体コードから算出）"),
    ("PREF_NAME", PREF_NAME, pref_name_source),
    ("CITY_NAME", CITY_NAME, "JSON"),
])))
print(f"  対象: {PREF_NAME}{CITY_NAME}")

In [ ]:
# ===== ABRマスター準備 =====
# ZIPアップロード→CSV種別判定→糸満市データ抽出→地番＋位置参照＋町字マスタの結合→住居表示データの
# 抽出・結合までの、一連の初期化処理をまとめて行う。ここで作ったtown_lg / parcel_master /
# residential_master等は、Colabランタイムが生きている間そのまま使い回せるため、住所CSVを差し替えるたびに
# このセルを再実行する必要はない（町字インデックス(TOWN_INDEX)の生成は、次の「住所変換ロジック定義」
# セルの末尾で行う）。

# --- ABR ZIPアップロード ---
# 地番住所用の3ZIP（mt_town_all / mt_parcel_city472107 / mt_parcel_pos_city472107）と、住居表示住所用の
# 4ZIP（mt_rsdtdsp_blk / mt_rsdtdsp_blk_pos / mt_rsdtdsp_rsdt / mt_rsdtdsp_rsdt_pos。沖縄県単位の配布）を
# まとめてアップロードする。ファイル名ではなく、ZIP内CSVのヘッダー列から種別を自動判定する。
# 不足しているデータを外部サイトから自動取得することはしない。

import io
import re
import unicodedata
import zipfile

import pandas as pd
from google.colab import files

# 種別判定に使う識別列（公式ABR CSVの列構成に基づく。列が全部揃っているものだけを該当種別とみなす）。
# ほかの種別の列を含み得るため、より限定的な種別から順に判定する:
#   住居表示-住居位置参照 → 住居表示-住居 → 住居表示-街区位置参照 → 住居表示-街区
#   → 地番位置参照(rep_lon/rep_lat) → 地番(prc_num1-3) → 町字
# （住居表示の各データは町字の列（oaza_cho等）も含み得るため、町字より先に見る）。
RSDT_POS_SIGNATURE = {"lg_code", "machiaza_id", "blk_id", "rsdt_id", "rsdt2_id", "rep_lon", "rep_lat"}
RSDT_SIGNATURE = {"lg_code", "machiaza_id", "blk_id", "rsdt_id", "rsdt2_id", "blk_num", "rsdt_num", "rsdt_num2"}
BLK_POS_SIGNATURE = {"lg_code", "machiaza_id", "blk_id", "rep_lon", "rep_lat"}
BLK_SIGNATURE = {"lg_code", "machiaza_id", "blk_id", "blk_num"}
PARCEL_POS_SIGNATURE = {"lg_code", "machiaza_id", "prc_id", "rep_lon", "rep_lat"}
PARCEL_SIGNATURE = {"lg_code", "machiaza_id", "prc_id", "prc_num1", "prc_num2", "prc_num3"}
TOWN_SIGNATURE = {"lg_code", "machiaza_id", "oaza_cho", "chome", "koaza", "machiaza_dist", "rsdt_addr_flg"}

# 種別ごとの表示名
ABR_KIND_LABELS = {
    "town": "町字マスタ",
    "parcel": "地番マスタ",
    "parcel_pos": "地番位置参照",
    "blk": "住居表示-街区",
    "blk_pos": "住居表示-街区位置参照",
    "rsdt": "住居表示-住居",
    "rsdt_pos": "住居表示-住居位置参照",
}


def classify_csv(columns):
    cols = set(columns)
    if RSDT_POS_SIGNATURE.issubset(cols):
        return "rsdt_pos"
    if RSDT_SIGNATURE.issubset(cols):
        return "rsdt"
    if BLK_POS_SIGNATURE.issubset(cols) and "rsdt_id" not in cols:
        return "blk_pos"
    if BLK_SIGNATURE.issubset(cols) and "rsdt_id" not in cols:
        return "blk"
    if PARCEL_POS_SIGNATURE.issubset(cols):
        return "parcel_pos"
    if PARCEL_SIGNATURE.issubset(cols):
        return "parcel"
    if TOWN_SIGNATURE.issubset(cols):
        return "town"
    if "post_code" in cols:
        return "post_code"
    return None


def read_csv_auto(file_bytes, **read_csv_kwargs):
    """UTF-8→CP932の順に読み込みを試みる（sheltermatch本体と同じ方針。Excelで保存したCSVも
    扱えるようにするため）。utf-8-sig はBOMがあれば取り除き、無ければ通常のUTF-8として読む。
    ABRマスターと住所CSVの両方から使う。"""
    last_error = None
    for encoding in ("utf-8-sig", "cp932"):
        try:
            return pd.read_csv(io.BytesIO(file_bytes), encoding=encoding, **read_csv_kwargs)
        except UnicodeDecodeError as e:
            last_error = e
    raise ValueError(f"文字コードを判定できなかった（UTF-8・CP932のいずれでも読み込めない）: {last_error}")


def read_abr_csv(fileobj):
    """ABR CSVを読み込む。ID・番号列の先頭ゼロや桁落ちを防ぐため全列を文字列として読み込み、
    欠損値は独自にNaN化せず空文字列のまま扱う（文字列連結・突合を単純にするため）。"""
    return read_csv_auto(fileobj.read(), dtype=str, keep_default_na=False, na_values=[])


print("次のABR ZIPをまとめて選択してください（ファイル名は例。種別はZIP内CSVの列構成で判定します）。")
print(f"  地番住所用（必須）: mt_town_all.csv.zip / mt_parcel_city{LG_CODE}.csv.zip / "
      f"mt_parcel_pos_city{LG_CODE}.csv.zip")
print("  住居表示住所用    : mt_rsdtdsp_blk_pref47.csv.zip / mt_rsdtdsp_blk_pos_pref47.csv.zip /")
print("                      mt_rsdtdsp_rsdt_pref47.csv.zip / mt_rsdtdsp_rsdt_pos_pref47.csv.zip")
print(f"  （住居表示データは沖縄県単位の配布データです。{CITY_NAME}（lg_code={LG_CODE}）の行だけを抽出して使います）")
uploaded_zips = files.upload()

# 種別ごとに読み込んだCSV（DataFrame）
abr_frames = {}

for zip_filename, zip_bytes in uploaded_zips.items():
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
        if not csv_members:
            print(f"'{zip_filename}' 内にCSVが見つからないため無視する。")
            continue
        for member in csv_members:
            with zf.open(member) as f:
                df = read_abr_csv(f)
            kind = classify_csv(df.columns)
            if kind in ABR_KIND_LABELS:
                print(f"'{zip_filename}' 内 '{member}' を{ABR_KIND_LABELS[kind]}として認識した。")
                abr_frames[kind] = df
            elif kind == "post_code":
                print(f"'{zip_filename}' 内 '{member}' は郵便番号データのため、今回の変換には使用しない。")
            else:
                print(f"'{zip_filename}' 内 '{member}' は既知の列構成と一致しないため無視する。")

town_df = abr_frames.get("town")
parcel_df = abr_frames.get("parcel")
parcel_pos_df = abr_frames.get("parcel_pos")

print()
print(f"町字マスタ    : {'OK' if town_df is not None else '不足'}")
print(f"地番マスタ    : {'OK' if parcel_df is not None else '不足'}")
print(f"地番位置参照  : {'OK' if parcel_pos_df is not None else '不足'}")
for kind in ("blk", "blk_pos", "rsdt", "rsdt_pos"):
    print(f"{ABR_KIND_LABELS[kind]}: {'OK' if kind in abr_frames else '未アップロード'}")

master_status = {}
master_status["zip_load"] = all(df is not None for df in (town_df, parcel_df, parcel_pos_df))

# --- 糸満市データ抽出 ---
# 町字マスタは全国データなのでlg_codeで抽出する。地番・位置参照も同じ自治体コードであることを確認する。

if not master_status.get("zip_load"):
    print("必要なABR CSVが揃っていないため、このセルはスキップする。")
    master_status["extract"] = False
else:
    town_lg = town_df[town_df["lg_code"] == LG_CODE].copy()
    parcel_lg = parcel_df[parcel_df["lg_code"] == LG_CODE].copy()
    parcel_pos_lg = parcel_pos_df[parcel_pos_df["lg_code"] == LG_CODE].copy()

    other_lg_in_parcel = (parcel_df["lg_code"] != LG_CODE).sum()
    other_lg_in_pos = (parcel_pos_df["lg_code"] != LG_CODE).sum()
    if other_lg_in_parcel:
        print(f"注意: 地番マスタに{LG_CODE}以外のlg_codeが{other_lg_in_parcel}件含まれている。")
    if other_lg_in_pos:
        print(f"注意: 地番位置参照に{LG_CODE}以外のlg_codeが{other_lg_in_pos}件含まれている。")

    print(f"{CITY_NAME}町字件数: {len(town_lg)}件")
    print(f"地番件数: {len(parcel_lg)}件")
    print(f"地番位置参照件数: {len(parcel_pos_lg)}件")

    print("\nrsdt_addr_flgの内訳（ABR上の住居表示フラグ。値の意味は独自解釈しない）:")
    print(town_lg["rsdt_addr_flg"].value_counts(dropna=False).to_string())

    town_lg["town_label"] = (
        town_lg["oaza_cho"] + town_lg["chome"] + town_lg["koaza"] + town_lg["machiaza_dist"]
    )

    residential_towns = town_lg[town_lg["rsdt_addr_flg"] == "1"]
    if len(residential_towns):
        print(f"\nrsdt_addr_flg=1（住居表示）の町字: {len(residential_towns)}件")
        for label in residential_towns["town_label"]:
            print(f"  - {label}")
    else:
        print("\nrsdt_addr_flg=1（住居表示）の町字はない。")

    master_status["extract"] = len(town_lg) > 0 and len(parcel_lg) > 0


# --- 地番＋座標マスター生成 ---
# 地番マスタ×地番位置参照(lg_code, machiaza_id, prc_id)、その結果×町字マスタ(lg_code, machiaza_id)で
# 結合する。キー重複がある場合はdrop_duplicates()で勝手に1件へ潰さず、validate=で結合関係を確認する。

if not master_status.get("extract"):
    print("糸満市データの抽出が完了していないため、このセルはスキップする。")
    master_status["merge"] = False
else:
    parcel_dupe = int(parcel_lg.duplicated(subset=["lg_code", "machiaza_id", "prc_id"]).sum())
    pos_dupe = int(parcel_pos_lg.duplicated(subset=["lg_code", "machiaza_id", "prc_id"]).sum())
    town_dupe = int(town_lg.duplicated(subset=["lg_code", "machiaza_id"]).sum())
    print(f"地番キー重複: {parcel_dupe}件")
    print(f"位置参照キー重複: {pos_dupe}件")
    print(f"町字キー重複: {town_dupe}件")

    try:
        parcel_master = parcel_lg.merge(
            parcel_pos_lg[["lg_code", "machiaza_id", "prc_id", "rep_lon", "rep_lat"]],
            on=["lg_code", "machiaza_id", "prc_id"],
            how="left",
            validate="one_to_one",
        )
        parcel_master = parcel_master.merge(
            town_lg[["lg_code", "machiaza_id", "town_label", "rsdt_addr_flg"]],
            on=["lg_code", "machiaza_id"],
            how="left",
            validate="many_to_one",
        )
        merge_ok = True
    except Exception as e:
        print(f"結合に失敗した（キー重複等の可能性がある）: {type(e).__name__}: {e}")
        parcel_master = None
        merge_ok = False

    master_status["merge"] = merge_ok
    if merge_ok:
        parcel_master["rep_lon"] = pd.to_numeric(parcel_master["rep_lon"], errors="coerce")
        parcel_master["rep_lat"] = pd.to_numeric(parcel_master["rep_lat"], errors="coerce")
        has_coords = parcel_master["rep_lon"].notna() & parcel_master["rep_lat"].notna()
        print(f"\n全地番件数: {len(parcel_master)}件")
        print(f"座標あり件数: {int(has_coords.sum())}件")
        print(f"座標なし件数: {int((~has_coords).sum())}件")


# --- 住居表示データの抽出・結合 ---
# 住居表示地域（町字マスタの rsdt_addr_flg=1）の住所は、住居表示-住居と住居表示-住居位置参照で
# 「町字＋街区符号＋住居番号（＋住居番号2）」まで一意に特定できた場合だけ座標を付ける。
# - 住居表示データは沖縄県単位で配布されるため、ファイル名（pref47等）ではなくCSV内の lg_code を正として、
#   糸満市（LG_CODE）の行だけを抽出して使う
# - 住所の座標は住居表示-住居位置参照からだけ取る。住居表示-街区位置参照は街区の代表点で、同じ街区に
#   複数の位置を持つこともあるため、座標には使わない（確認用に件数を表示するだけ）
# - 住居と住居位置参照は RSDT_JOIN_KEYS で1対1に対応することを確認してから結合する。キーが一意でない
#   場合は drop_duplicates() 等で勝手に1件へ潰さず、処理を止める
# 住居表示データを1つもアップロードしなかった場合は、住居表示地域の住所を照合せず
# residential_display_area として保留する（地番住所の変換はそのまま行える）。

RESIDENTIAL_KINDS = ("blk", "blk_pos", "rsdt", "rsdt_pos")
# 住居表示の照合に必要なデータ（住居表示-街区位置参照は確認用で、照合には使わない）
RESIDENTIAL_REQUIRED_KINDS = ("blk", "rsdt", "rsdt_pos")
RSDT_JOIN_KEYS = ["lg_code", "machiaza_id", "rsdt_addr_flg", "blk_id", "rsdt_id", "rsdt2_id"]
BLK_POS_KEYS = ["lg_code", "machiaza_id", "blk_id", "rsdt_addr_flg"]


def build_residential_master(rsdt_lg, rsdt_pos_lg):
    """住居表示-住居と住居表示-住居位置参照を RSDT_JOIN_KEYS で結合し、住居ごとに rep_lon / rep_lat を
    持つ表を返す。rsdt2_id が空欄の住居も、空欄同士として対応付ける（空文字列のまま比較する）。
    キーの列が無い・キーが一意でない場合は、住居と座標を1対1に対応付けられないため例外を送出する
    （どれか1件を選んで座標を付けることはしない）。住居位置参照が見つからない住居は、座標なし
    （照合すると coordinates_missing）として残す。"""
    for kind, df in (("rsdt", rsdt_lg), ("rsdt_pos", rsdt_pos_lg)):
        missing_keys = [c for c in RSDT_JOIN_KEYS if c not in df.columns]
        if missing_keys:
            raise RuntimeError(
                f"{ABR_KIND_LABELS[kind]}に、住居と住居位置参照を対応付けるための列がありません: "
                f"{', '.join(missing_keys)}。取得したデータを確認してください。"
            )

    rsdt_dupe = int(rsdt_lg.duplicated(subset=RSDT_JOIN_KEYS).sum())
    pos_dupe = int(rsdt_pos_lg.duplicated(subset=RSDT_JOIN_KEYS).sum())
    print(f"住居マスターのキー重複: {rsdt_dupe}件")
    print(f"住居位置参照のキー重複: {pos_dupe}件")
    if rsdt_dupe or pos_dupe:
        raise RuntimeError(
            "住居表示-住居／住居表示-住居位置参照のキー（" + ", ".join(RSDT_JOIN_KEYS) + "）が一意でないため、"
            "住居と座標を1対1に対応付けられません。どれか1件を選んで座標を付けることはしないため、"
            "処理を中止します。取得したデータを確認してください。"
        )

    merged = rsdt_lg.merge(
        rsdt_pos_lg[RSDT_JOIN_KEYS + ["rep_lon", "rep_lat"]],
        on=RSDT_JOIN_KEYS,
        how="left",
        validate="one_to_one",
        indicator="_position",
    )
    joined_count = int((merged["_position"] == "both").sum())
    merged = merged.drop(columns="_position")
    merged["rep_lon"] = pd.to_numeric(merged["rep_lon"], errors="coerce")
    merged["rep_lat"] = pd.to_numeric(merged["rep_lat"], errors="coerce")
    has_coords = merged["rep_lon"].notna() & merged["rep_lat"].notna()

    print(f"住居マスターと住居位置参照の結合: {joined_count}件（住居マスター {len(rsdt_lg)}件中）")
    print(f"住居マスターに対して位置参照が見つからない: {len(rsdt_lg) - joined_count}件")
    print(f"住居マスターに対応しない住居位置参照（照合には使わない）: {len(rsdt_pos_lg) - joined_count}件")
    print(f"座標欠損（住居位置参照はあるが緯度・経度が無い）: {joined_count - int(has_coords.sum())}件")
    return merged


residential_master = None
residential_blocks = set()
residential_lg = {}
uploaded_residential_kinds = [kind for kind in RESIDENTIAL_KINDS if kind in abr_frames]

if not master_status.get("merge"):
    master_status["residential"] = "skipped"
elif not uploaded_residential_kinds:
    master_status["residential"] = "not_uploaded"
    print("\n住居表示データはアップロードされていない。住居表示地域（rsdt_addr_flg=1）の住所は照合せず、"
          "residential_display_area として保留する。")
else:
    # 最後まで確認できたときだけ "loaded" にする（途中で止まった場合、住所CSV変換セルは実行しない）
    master_status["residential"] = "error"
    missing_kinds = [kind for kind in RESIDENTIAL_REQUIRED_KINDS if kind not in abr_frames]
    if missing_kinds:
        raise RuntimeError(
            "住居表示データが揃っていないため、住居表示住所の座標を正しく作れません: "
            + "、".join(ABR_KIND_LABELS[kind] for kind in missing_kinds) + " がありません。"
            "住居表示データ（mt_rsdtdsp_*）をまとめてアップロードし直してください。"
        )

    print(f"\n住居表示データ（沖縄県単位の配布データ）から、{CITY_NAME}（lg_code={LG_CODE}）の行だけを抽出した。")
    for kind in uploaded_residential_kinds:
        df = abr_frames[kind]
        residential_lg[kind] = df[df["lg_code"] == LG_CODE].copy()
        print(f"{ABR_KIND_LABELS[kind]}: {len(residential_lg[kind])}件（ファイル全体 {len(df)}件）")

    empty_kinds = [kind for kind in RESIDENTIAL_REQUIRED_KINDS if len(residential_lg[kind]) == 0]
    if empty_kinds:
        raise RuntimeError(
            "、".join(ABR_KIND_LABELS[kind] for kind in empty_kinds)
            + f" に{CITY_NAME}（lg_code={LG_CODE}）の行がありません。対象の都道府県のデータか確認してください。"
        )

    # 住居表示-街区位置参照は確認用。同じ街区に複数の位置があっても住所の座標には使わないため、異常として扱わない
    blk_pos_lg = residential_lg.get("blk_pos")
    if blk_pos_lg is not None and all(c in blk_pos_lg.columns for c in BLK_POS_KEYS):
        multi_position = blk_pos_lg.duplicated(subset=BLK_POS_KEYS, keep=False)
        multi_block_count = blk_pos_lg[multi_position].groupby(BLK_POS_KEYS).ngroups
        print(f"（参考）街区位置参照で、同じ街区に複数の位置がある街区: {multi_block_count}街区"
              f"（{int(multi_position.sum())}行）。街区位置参照は住所の座標に使わないため、異常として扱わない。")

    residential_master = build_residential_master(residential_lg["rsdt"], residential_lg["rsdt_pos"])
    # 街区の有無（住居番号まで見つからなかったときに、街区が見つからないのか住居番号が見つからないのかを
    # 区別するためだけに使う。街区の座標は使わない）
    residential_blocks = (
        set(zip(residential_lg["blk"]["machiaza_id"], residential_lg["blk"]["blk_num"]))
        | set(zip(residential_master["machiaza_id"], residential_master["blk_num"]))
    )
    master_status["residential"] = "loaded"

if master_status.get("merge"):
    print("\nABRマスター準備が完了しました。次の「住所変換ロジック定義」セルへ進んでください。")
    if master_status["residential"] == "loaded":
        print("住居表示データも読み込んだため、住居表示地域の住所も住居番号まで照合します。")
else:
    print("\nABRマスター準備に失敗しました。住所CSV変換セルは実行できません。上の出力を確認してください。")


In [ ]:
# ===== 住所変換ロジック定義 =====
# 正規化→町字インデックス作成・照合→地番／住居表示の番号解析・照合→方書を除いた再照合→
# geocode_one_address() までの、住所→座標変換ロジックをまとめて定義する
# （PoC: experiments/abr_csv_geocode_poc.ipynb で実機確認したロジックが出発点。
#   その後の表記揺れ対応はこちらに入っている。回帰テストは test/test_address_conversion.py）。

# --- 住所正規化関数 ---
# PoCとして必要十分な範囲のみ正規化する。住所正規化ライブラリの新規導入や、
# 巨大な独自住所パーサーの実装はしない。

HYPHEN_CHARS = "－ー‐‑–—―─−"
HYPHEN_TRANS = str.maketrans({c: "-" for c in HYPHEN_CHARS})

# 丁目の漢数字（例:「西崎町一丁目」）。ABR側は算用数字のため、丁目の直前にある漢数字だけを
# 算用数字へ揃える。地番の漢数字（「六七三番地」等）は桁の解釈が一意に決まらないため変換しない。
KANJI_DIGITS = {"一": 1, "二": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9}
KANJI_CHOME_PATTERN = re.compile(r"[一二三四五六七八九十]+(?=丁目)")


def kanji_to_number(text):
    """「一」〜「九十九」の漢数字を数値にする。解釈できない並びはNoneを返す。"""
    tens, separator, ones = text.partition("十")
    if not separator:
        return KANJI_DIGITS.get(text)
    tens_value = 1 if tens == "" else KANJI_DIGITS.get(tens)
    ones_value = 0 if ones == "" else KANJI_DIGITS.get(ones)
    if tens_value is None or ones_value is None:
        return None
    return tens_value * 10 + ones_value


def normalize_kanji_chome(text):
    """「一丁目」を「1丁目」へ揃える。解釈できない並びはそのまま残す。"""
    def replace(match):
        number = kanji_to_number(match.group())
        return match.group() if number is None else str(number)
    return KANJI_CHOME_PATTERN.sub(replace, text)


def _remove_space(match):
    """空白を取り除く。ただし数字と数字の間の空白は1つの半角空白として残す（「673-2 101」を詰めて
    「673-2101」という別の番号にしないため。残った空白は番号として読めないため、その住所は照合されない）。"""
    text, start, end = match.string, match.start(), match.end()
    between_digits = 0 < start and end < len(text) and text[start - 1].isdigit() and text[end].isdigit()
    return " " if between_digits else ""


def normalize_address(text):
    """Unicode NFKC正規化（全角数字→半角等）、ハイフン類の統一、空白除去（数字と数字の間の空白を除く）、
    丁目の漢数字→算用数字を行う。"""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(HYPHEN_TRANS)
    text = re.sub(r"\s+", _remove_space, text)
    return normalize_kanji_chome(text)


def strip_pref_city(text):
    """先頭の都道府県名・市区町村名を取り除く（無くても動くようにする）。"""
    for prefix in (PREF_NAME + CITY_NAME, CITY_NAME, PREF_NAME):
        if text.startswith(prefix):
            return text[len(prefix):]
    return text


print("normalize_address() / strip_pref_city() を定義した。")
print("例:", strip_pref_city(normalize_address(" 沖縄県糸満市字糸満６７３ ")))


# --- 町字判定 ---
# oaza_cho+chome+koaza+machiaza_distで町字表記を作り、入力住所の先頭と最も長く一致する町字を採用する。
# ABR原文は全角数字（例:「１丁目」）を含み得るため、検索キーには入力住所と同じnormalize_address()を
# 適用する（表示用のofficial_labelはABR原文のまま保持する）。沖縄県でよくある「字」の有無や
# 「大字」表記は、正式名称は保持したまま、検索用の別名として許容する。ただし別名を作るのは
# ABRの正式名称が「字」「大字」で始まる場合だけで、「西崎町1丁目」のような町字に
# 「字西崎町1丁目」という存在しない別名は作らない。
# 同じ長さで複数machiaza_idに一致する場合はambiguous_townとする。


TOWN_NAME_PREFIXES = ("大字", "字")


def town_search_keys(search_key):
    """1つの町字表記から、検索用の別名を作る。入力住所では「字」が省略されたり「大字」と
    書かれたりするため、ABRの正式名称が「字」「大字」で始まる場合に限り、接頭辞を外した形と
    付け替えた形も検索キーとして許容する（正式名称 official_label は変更しない）。
    「西崎町1丁目」のように接頭辞の無い正式名称には、別名を作らない
    （実際の表記揺れを超えて、存在しない住所を既存の町字として確定させないため）。"""
    for prefix in TOWN_NAME_PREFIXES:
        if search_key.startswith(prefix) and len(search_key) > len(prefix):
            bare = search_key[len(prefix):]
            return list(dict.fromkeys([search_key, bare, "字" + bare, "大字" + bare]))
    return [search_key]


def build_town_index(town_lg):
    """町字マスタから検索用インデックスを作る。ABR原文の表記（全角数字等を含み得る）を
    official_labelとしてそのまま保持しつつ、search_keyは入力住所と同じnormalize_address()を
    通して作る（入力側だけを正規化して原文のままの町字と比較すると、全角丁目表記等で
    一致しなくなるため）。"""
    entries = []
    for _, row in town_lg.iterrows():
        label = row["town_label"]
        if not label:
            continue
        search_key = normalize_address(label)
        if not search_key:
            continue
        for key in town_search_keys(search_key):
            entries.append(
                {"search_key": key, "official_label": label, "machiaza_id": row["machiaza_id"],
                 "rsdt_addr_flg": row["rsdt_addr_flg"], "is_official_name": key == search_key}
            )
    return entries


def match_town(remainder, town_index):
    """remainderの先頭と最も長く一致する町字を返す。
    戻り値は (一致したエントリ or None, 一致部分を除いた残り文字列, 曖昧かどうか)。

    同じ長さで複数の町字に一致した場合、ABRの正式名称そのものに一致したものがあれば
    それを優先する（例: 「字上里」と「大字上里」が別の町字として存在する場合、
    「字上里…」という入力は正式名称「字上里」の町字とみなす）。正式名称に一致するものが
    無く、接頭辞を変換した別名だけで複数の町字に当たる場合（例:「上里…」）は、
    どちらか判断できないためambiguous_townとする。"""
    best_len = -1
    candidates = []
    for entry in town_index:
        key = entry["search_key"]
        if key and remainder.startswith(key):
            if len(key) > best_len:
                best_len = len(key)
                candidates = [entry]
            elif len(key) == best_len:
                candidates.append(entry)

    if not candidates:
        return None, remainder, False

    official_matches = [c for c in candidates if c["is_official_name"]]
    if official_matches:
        candidates = official_matches

    distinct_ids = {c["machiaza_id"] for c in candidates}
    if len(distinct_ids) > 1:
        return None, remainder, True

    return candidates[0], remainder[best_len:], False


if master_status.get("extract"):
    TOWN_INDEX = build_town_index(town_lg)
    print(f"町字インデックスを{len(TOWN_INDEX)}件（別名込み）作成した。")
else:
    TOWN_INDEX = []
    print("糸満市データの抽出が完了していないため、町字インデックスは作成しない。")


# --- 地番判定 ---
# 町字確定後の残り文字列からprc_num1/2/3を取得し、地番マスタと(lg_code, machiaza_id,
# prc_num1, prc_num2, prc_num3)で照合する。前方一致だけで座標を決定しない
# （例: "673" と "673-2" は別地番として扱う）。候補が複数残る場合はambiguous_parcelとする。


# 地番の区切りに使われる表記。いずれも「削除」ではなく区切り文字'-'へ置き換える。
# 削除してしまうと「673番地2」が「6732」となり、別の地番へ誤って一致する恐れがあるため。
PARCEL_SEPARATORS = re.compile(r"番地の|番の|番地|番|号|の")


def parse_parcel_numbers(text):
    """'673' '673番地' '673-2' '673番地の2' '673番地2' 等からprc_num1〜3を抽出する。
    数字として解釈できない場合や、4つ以上の要素に分かれる場合はNoneを返す
    （prc_num1〜3の3要素までしか無いため、黙って切り詰めて別地番へ誤一致させない）。"""
    if not text:
        return None

    parts = [p for p in PARCEL_SEPARATORS.sub("-", text.strip()).split("-") if p != ""]
    if not parts or len(parts) > 3 or any(not p.isdigit() for p in parts):
        return None

    return tuple((parts + ["", ""])[:3])


def match_parcel(machiaza_id, numbers, parcel_master):
    n1, n2, n3 = numbers
    subset = parcel_master[parcel_master["machiaza_id"] == machiaza_id]
    subset = subset[
        (subset["prc_num1"] == n1) & (subset["prc_num2"] == n2) & (subset["prc_num3"] == n3)
    ]
    return subset


print("parse_parcel_numbers() / match_parcel() を定義した。")
for example in ("673", "673番地", "673番地の2", "673番地2"):
    print(f"  例: {example} -> {parse_parcel_numbers(example)}")


# --- 住居表示判定 ---
# 住居表示地域（町字のrsdt_addr_flg==1）では、町字確定後の残り文字列から街区符号・住居番号・住居番号2
# （ABRの blk_num / rsdt_num / rsdt_num2）を読み取り、住居表示-住居（住居位置参照と結合済み）と
# (machiaza_id, blk_num, rsdt_num, rsdt_num2) の完全一致で照合する。丁目は町字側で判定済み
# （例:「○○2丁目32番2号」→ 街区32・住居番号2）。一致が1件だけの場合に限り、住居位置参照の座標を使う。
# 街区だけが一致した場合に街区の代表点を使ったり、近い住居番号へ寄せたりはしない。
#
# 3段の番号（例:「36番7-201号」→ 36・7・201）は、まず3段のまま（201を住居番号2として）完全一致を試す。
# 3段目を部屋番号等の付加情報とみなして除外する（_match_without_last_number）のは、次をすべて満たす場合だけ:
#   - 元の住所から空白区切りの方書（建物名等）を実際に除外した候補を照合している
#     （allow_last_number_fallback=True。方書の無い「19番14-2号」は、3段目が部屋番号なのかABRに未収録の
#     住居番号2なのかを確定できないため、除外しない）
#   - 3段の完全一致が0件
#   - 同じ町字・街区符号・住居番号で、住居番号2が空欄の住居がちょうど1件ある
#   - その住居の住居位置参照に座標がある
#   - 同じ町字・街区符号・住居番号で、住居番号2を持つ住居がABRに無い（ある場合、3段目は部屋番号ではなく
#     ABRに無い住居番号2の可能性があるため、近い住居へ寄せないよう除外しない）
# 3段目を最初から部屋番号と決めつけることはしない。座標は避難所候補の距離順位に使われるため、確定できない
# 場合は推測で一致させず、未変換として残す。


def parse_residential_numbers(text):
    """'37番14号' '37-14' '37番14' '37番14号の1' 等から (街区符号, 住居番号, 住居番号2) を読み取る。
    区切りの表記は地番と同じ規則で扱う（parse_parcel_numbers）。番号を補完・近似・切り捨てはせず、
    読み取れない場合や4つ以上の要素に分かれる場合はNoneを返す。"""
    return parse_parcel_numbers(text)


def match_residential(machiaza_id, numbers, residential_master):
    blk_num, rsdt_num, rsdt_num2 = numbers
    subset = residential_master[residential_master["machiaza_id"] == machiaza_id]
    subset = subset[
        (subset["blk_num"] == blk_num) & (subset["rsdt_num"] == rsdt_num) & (subset["rsdt_num2"] == rsdt_num2)
    ]
    return subset


print("parse_residential_numbers() / match_residential() を定義した。")
for example in ("37番14号", "37-14", "37番14号の1"):
    print(f"  例: {example} -> {parse_residential_numbers(example)}")


# --- 1件の住所の照合（_geocode_address_core） ---
# 正規化→町字判定→（rsdt_addr_flgが1以外）地番照合／（1）住居表示照合 の1回分。方書の除外はここでは
# 行わない。住居表示データを読み込んでいない場合、住居表示地域の住所は照合せず
# residential_display_area として保留する（地番マスタで似た番号が見つかっても採用しない）。

# abr_address: ABRで一致した住所（町字＋地番、または町字＋街区符号・住居番号。matched のときだけ）
# ignored_number: 住居表示の3段目を部屋番号等の付加情報として除外した場合の、除外した番号
RESULT_FIELDS = (
    "normalized_address", "matched_town", "machiaza_id", "address_type",
    "parcel_number", "prc_id", "blk_num", "rsdt_num", "rsdt_num2",
    "longitude", "latitude", "rsdt_addr_flg", "matched_address", "ignored_suffix",
    "abr_address", "ignored_number",
)


def _result(input_address, status, **fields):
    unknown = set(fields) - set(RESULT_FIELDS)
    if unknown:
        raise TypeError(f"未知の項目: {sorted(unknown)}")
    result = {"input_address": input_address}
    result.update({field: fields.get(field) for field in RESULT_FIELDS})
    result["status"] = status
    return result


def _match_parcel_address(address, town_remainder, common_kwargs):
    """地番住所の照合（従来の地番判定）。"""
    common_kwargs = dict(common_kwargs, address_type="地番")
    numbers = parse_parcel_numbers(town_remainder)
    if numbers is None:
        return _result(address, "parcel_not_found", **common_kwargs)

    parcel_number_display = "-".join(p for p in numbers if p)
    matches = match_parcel(common_kwargs["machiaza_id"], numbers, PARCEL_MASTER)

    if len(matches) == 0:
        return _result(address, "parcel_not_found", parcel_number=parcel_number_display, **common_kwargs)
    if len(matches) > 1:
        return _result(address, "ambiguous_parcel", parcel_number=parcel_number_display, **common_kwargs)

    row = matches.iloc[0]
    if pd.isna(row["rep_lon"]) or pd.isna(row["rep_lat"]):
        return _result(
            address, "coordinates_missing", parcel_number=parcel_number_display,
            prc_id=row["prc_id"], **common_kwargs
        )

    return _result(
        address, "matched", parcel_number=parcel_number_display, prc_id=row["prc_id"],
        longitude=row["rep_lon"], latitude=row["rep_lat"],
        abr_address=f"{common_kwargs['matched_town']}{parcel_number_display}", **common_kwargs
    )


def residential_abr_address(matched_town, blk_num, rsdt_num, rsdt_num2):
    """ABRで一致した住居表示住所の表示用文字列（確認用。例: '架空台２丁目36番7号'）。"""
    number = f"{rsdt_num}-{rsdt_num2}" if rsdt_num2 else rsdt_num
    return f"{matched_town}{blk_num}番{number}号"


def _match_without_last_number(address, common_kwargs):
    """方書を除外した候補の照合で、3段の番号（街区符号・住居番号・3段目）が住居表示-住居に完全一致
    しなかった場合だけ呼ぶ（住所全文の照合からは呼ばない）。
    3段目を部屋番号等の付加情報とみなせる場合（冒頭のコメントの条件をすべて満たす場合）に限り、3段目を
    除いた住居で matched を返す。2段の住居が複数ある場合は ambiguous_residential を返す。
    それ以外（2段の住居が無い・座標が無い・住居番号2を持つ住居がある）は None を返し、呼び出し側で
    従来どおり未変換として扱う。"""
    subset = RESIDENTIAL_MASTER[RESIDENTIAL_MASTER["machiaza_id"] == common_kwargs["machiaza_id"]]
    same_number = subset[
        (subset["blk_num"] == common_kwargs["blk_num"]) & (subset["rsdt_num"] == common_kwargs["rsdt_num"])
    ]
    if (same_number["rsdt_num2"] != "").any():
        return None  # 住居番号2を使っている住居番号のため、3段目は部屋番号とみなさない
    if len(same_number) > 1:
        return _result(address, "ambiguous_residential", **common_kwargs)
    if len(same_number) == 0:
        return None

    row = same_number.iloc[0]
    if pd.isna(row["rep_lon"]) or pd.isna(row["rep_lat"]):
        return None

    ignored_number = common_kwargs["rsdt_num2"]
    matched_kwargs = dict(common_kwargs, rsdt_num2="")
    return _result(
        address, "matched", longitude=row["rep_lon"], latitude=row["rep_lat"], ignored_number=ignored_number,
        abr_address=residential_abr_address(
            common_kwargs["matched_town"], common_kwargs["blk_num"], common_kwargs["rsdt_num"], ""),
        **matched_kwargs
    )


def _match_residential_address(address, town_remainder, common_kwargs, allow_last_number_fallback=False):
    """住居表示住所の照合。町字＋街区符号＋住居番号（＋住居番号2）が住居表示-住居の1件に完全一致し、
    住居位置参照の座標がある場合だけ matched とする。allow_last_number_fallback=True（方書を除外した候補の
    照合）で、3段の番号が完全一致しない場合に限り、3段目を部屋番号等として除外できるかを確認する
    （_match_without_last_number）。"""
    common_kwargs = dict(common_kwargs, address_type="住居表示")
    if RESIDENTIAL_MASTER is None:
        return _result(address, "residential_display_area", **common_kwargs)

    numbers = parse_residential_numbers(town_remainder)
    if numbers is None:
        return _result(address, "residential_block_not_found", **common_kwargs)

    blk_num, rsdt_num, rsdt_num2 = numbers
    common_kwargs = dict(common_kwargs, blk_num=blk_num, rsdt_num=rsdt_num, rsdt_num2=rsdt_num2)
    matches = match_residential(common_kwargs["machiaza_id"], numbers, RESIDENTIAL_MASTER)

    if len(matches) == 0 and rsdt_num2 and allow_last_number_fallback:
        without_last_number = _match_without_last_number(address, common_kwargs)
        if without_last_number is not None:
            return without_last_number

    if len(matches) == 0:
        # 街区まで一致しても、住居番号まで一致しなければ座標は付けない（街区の代表点・近い番号は使わない）
        block_exists = (common_kwargs["machiaza_id"], blk_num) in RESIDENTIAL_BLOCKS
        status = "residential_number_not_found" if block_exists else "residential_block_not_found"
        return _result(address, status, **common_kwargs)
    if len(matches) > 1:
        return _result(address, "ambiguous_residential", **common_kwargs)

    row = matches.iloc[0]
    if pd.isna(row["rep_lon"]) or pd.isna(row["rep_lat"]):
        return _result(address, "coordinates_missing", **common_kwargs)

    return _result(
        address, "matched", longitude=row["rep_lon"], latitude=row["rep_lat"],
        abr_address=residential_abr_address(common_kwargs["matched_town"], blk_num, rsdt_num, rsdt_num2),
        **common_kwargs
    )


def _geocode_address_core(address, allow_last_number_fallback=False):
    """方書の除外を行わない、1回分の住所照合。address は照合に使う文字列（住所全文、または方書を
    除いた候補）。allow_last_number_fallback は、住居表示の3段目を部屋番号等として除外することを
    許可するか（空白区切りの方書を実際に除外した候補を照合するときだけ True にする）。"""
    normalized = normalize_address(address)
    remainder = strip_pref_city(normalized)

    town_entry, town_remainder, town_ambiguous = match_town(remainder, TOWN_INDEX)
    if town_ambiguous:
        return _result(address, "ambiguous_town", normalized_address=normalized)
    if town_entry is None:
        return _result(address, "town_not_found", normalized_address=normalized)

    common_kwargs = dict(
        normalized_address=normalized,
        matched_town=town_entry["official_label"],
        machiaza_id=town_entry["machiaza_id"],
        rsdt_addr_flg=town_entry["rsdt_addr_flg"],
    )
    if town_entry["rsdt_addr_flg"] == "1":
        return _match_residential_address(address, town_remainder, common_kwargs, allow_last_number_fallback)
    return _match_parcel_address(address, town_remainder, common_kwargs)


# --- 方書の除外・geocode_one_address() ---
# 住所の後ろに施設名・建物名・部屋番号等の方書が付いていると、番号として読めず照合できない。そこで、
# まず住所全文で照合し、「見つからない」だった場合だけ、元の住所（正規化前）にある空白の位置で
# 右から順に後ろを切り落とした候補で照合し直す（例:「…927番地の2 架空ホーム 東棟」→
# 「…927番地の2 架空ホーム」→「…927番地の2」）。
# - 候補はABRと完全一致（matched）した場合だけ採用する。方書の除外で曖昧さを解消することはしない
# - 空白の無い住所は切り詰めない（「673-2-101」を「673-2」にする等、末尾を部屋番号と推測しない）。
#   空白があっても、数字だけを切り落とすことになる位置（「673-2 101」等）では切らない
# - 住居表示住所の3段目（「36番7-201号」の201）は、方書を除外した候補を照合するときだけ、上の
#   「住居表示判定」の条件を満たす場合に除外する（住所全文の照合では除外しない）
# - 住所の文字列（address列）は変更しない。方書を除くのは照合のときだけ
# - 曖昧（ambiguous_*）・座標欠損・住居表示データ未読込の場合は、方書を除いても正しい結果に
#   ならないため再照合しない

RETRY_WITHOUT_SUFFIX_STATUSES = {"parcel_not_found", "residential_block_not_found", "residential_number_not_found"}
# 方書を除いた候補がこれらの状態なら、さらに短い候補へ進む（町字まで切り落とした候補を含む）
CONTINUE_SUFFIX_STATUSES = RETRY_WITHOUT_SUFFIX_STATUSES | {"town_not_found"}


def suffix_candidates(raw_address):
    """元の住所にある空白の位置で、右から順に後ろを切り落とした候補の一覧を返す。
    数字と数字の間の空白で、切り落とす部分の先頭の語が数字だけの場合（例:「673 2」の「2」、
    「673-2 101」の「101」）は、番号の続きか部屋番号か判断できないため、その位置では切らない
    （「101号室」のように方書と分かる場合は切る）。"""
    text = str(raw_address).strip()
    candidates = []
    for match in reversed(list(re.finditer(r"\s+", text))):
        left, removed = text[:match.start()], text[match.end():]
        first_word = normalize_address(removed.split()[0])
        if normalize_address(left[-1:]).isdigit() and re.fullmatch(r"[\d-]+", first_word):
            continue
        candidates.append(left)
    return candidates


def geocode_one_address(raw_address):
    if pd.isna(raw_address) or not str(raw_address).strip():
        return _result(raw_address, "blank_address")

    text = str(raw_address).strip()
    # 住所全文の照合では、住居表示の3段目を部屋番号等として除外しない
    result = _geocode_address_core(text, allow_last_number_fallback=False)
    used = text
    if result["status"] in RETRY_WITHOUT_SUFFIX_STATUSES:
        # 見つからなかった場合の表示用に、番号まで読み取れた最初の結果を残しておく
        fallback = (result, used) if (result["parcel_number"] or result["blk_num"]) else None
        for candidate in suffix_candidates(text):
            # 空白区切りの方書を実際に除外した候補なので、住居表示の3段目の除外を許可する
            # （除外できるかどうかは _match_without_last_number の条件で判断する）
            candidate_result = _geocode_address_core(candidate, allow_last_number_fallback=True)
            if candidate_result["status"] not in CONTINUE_SUFFIX_STATUSES:
                result, used = candidate_result, candidate
                break
            if fallback is None and (candidate_result["parcel_number"] or candidate_result["blk_num"]):
                fallback = (candidate_result, candidate)
        else:
            if fallback is not None:
                result, used = fallback

    result["input_address"] = raw_address
    result["matched_address"] = used
    result["ignored_suffix"] = text[len(used):].strip()
    return result


if master_status.get("merge"):
    PARCEL_MASTER = parcel_master
else:
    PARCEL_MASTER = None

if master_status.get("residential") == "loaded":
    RESIDENTIAL_MASTER = residential_master
    RESIDENTIAL_BLOCKS = residential_blocks
else:
    RESIDENTIAL_MASTER = None
    RESIDENTIAL_BLOCKS = set()

if master_status.get("merge"):
    print("geocode_one_address() を定義した（町字判定→地番照合／住居表示照合→方書を除いた再照合の順で処理する）。")
    if RESIDENTIAL_MASTER is None:
        print("住居表示データが読み込まれていないため、住居表示地域の住所は residential_display_area として保留する。")
else:
    print("地番・位置参照の結合が完了していないため、geocode_one_address()は定義のみ行う。")


In [ ]:
# ===== 住所CSV変換・出力 =====
# 共通住民CSV（resident_id,address,latitude,longitude,geocode_status の5列）をアップロード→検証→変換→
# 結果確認→CSV保存→ダウンロードまでを一度に行う。ABRマスター・住所変換ロジック（前の2セル）はそのままで、
# このセルだけを何度でも再実行できる。出力CSVは入力と同じ5列のみを持つ（normalized_address等の
# ABR内部確認用の列は含めない。Notebook上の確認表示にのみ使う）。latitude/longitude/geocode_statusは
# 常に最新の変換結果で上書きするため、住所を修正して同じCSVを再投入した場合も古い座標・状態は残らない。
# address列は入力のまま出力する（方書を除くのは照合のときだけで、address列は書き換えない）。

from pathlib import Path

# 共通住民CSVの5列。入力の必須列であり、出力CSVの列でもある。
RESIDENT_COLUMNS = ["resident_id", "address", "latitude", "longitude", "geocode_status"]
# ABR内部確認用の列。正式運用CSVには含めず、Notebook上の確認表示にのみ使う。
# address_type: 地番／住居表示、matched_address: 照合に使った住所（方書を除いた場合はその残り）、
# ignored_suffix: 照合のときに除いた方書、blk_num / rsdt_num / rsdt_num2: 住居表示の街区符号・住居番号・住居番号2、
# abr_address: ABRで一致した住所（matched のとき）、ignored_number: 住居表示の3段目を部屋番号等として除外した番号
DEBUG_COLUMNS = ["normalized_address", "matched_town", "machiaza_id", "parcel_number", "prc_id", "rsdt_addr_flg",
                 "address_type", "blk_num", "rsdt_num", "rsdt_num2", "matched_address", "ignored_suffix",
                 "abr_address", "ignored_number"]


def find_resident_id_problems(df):
    """resident_id列の空欄・重複を検出する。resident_idは住所変換結果と避難所候補算出結果を
    紐づける結合キーのため、曖昧な状態のまま処理を進めない。戻り値は問題を説明する文字列のリスト
    （問題が無ければ空リスト）。行番号は職員が見るCSVの行番号（見出しを1行目として数えるため、
    0始まりの行位置に2を足した値）で示す。"""
    problems = []
    resident_id = df["resident_id"]
    blank_mask = resident_id.isna() | (resident_id.astype(str).str.strip() == "")
    duplicate_mask = resident_id.duplicated(keep=False) & ~blank_mask

    blank_rows = [position + 2 for position, blank in enumerate(blank_mask) if blank]
    if blank_rows:
        problems.append(f"resident_id が空欄の行があります（CSV行番号: {', '.join(map(str, blank_rows))}）")

    duplicate_detail = [
        f"CSV行{position + 2}='{value}'"
        for position, (value, duplicated) in enumerate(zip(resident_id, duplicate_mask))
        if duplicated
    ]
    if duplicate_detail:
        problems.append(f"resident_id が重複している行があります（{', '.join(duplicate_detail)}）")

    return problems


def build_geocoded_filename(csv_filename):
    """入力ファイル名から出力ファイル名を作る。出力を再度入力として使う運用を想定し、
    既にstem末尾が'_geocoded'の場合は追加しない（冪等にする。例:
    residents.csv → residents_geocoded.csv、residents_geocoded.csv → residents_geocoded.csv）。"""
    stem = Path(csv_filename).stem
    if stem.endswith("_geocoded"):
        return f"{stem}.csv"
    return f"{stem}_geocoded.csv"


if not master_status.get("merge"):
    print("ABRマスターの準備が完了していないため、このセルはスキップする。先に「ABRマスター準備」セルを実行すること。")
elif master_status.get("residential") == "error":
    print("住居表示データを正しく準備できなかったため、このセルはスキップする。「ABRマスター準備」セルの出力を"
          "確認し、住居表示データをアップロードし直してから再実行すること。")
else:
    if RESIDENTIAL_MASTER is None:
        print("住居表示データ: 未読込（住居表示地域の住所は residential_display_area として保留する）")
    else:
        print("住居表示データ: 読込済み（住居表示地域の住所も住居番号まで照合する）")
    print("resident_id,address,latitude,longitude,geocode_status の5列を持つ共通住民CSVを選択してください")
    print("（初回変換時は latitude / longitude / geocode_status が空欄でも構いません）。")
    uploaded_csv = files.upload()

    if not uploaded_csv:
        print("CSVがアップロードされなかったため、このセルはスキップされた。")
    elif len(uploaded_csv) > 1:
        print("住所CSVは1つだけ選択してください。")
    else:
        csv_filename = list(uploaded_csv.keys())[0]
        # resident_id・addressを入力の表記のまま保つため、全列を文字列として読み込む
        # （先頭ゼロを消さず、「N/A」等の文字も欠損値へ置き換えない）。
        addresses_df = read_csv_auto(uploaded_csv[csv_filename], dtype=str, keep_default_na=False)

        missing_columns = [c for c in RESIDENT_COLUMNS if c not in addresses_df.columns]
        if missing_columns:
            print(f"必須列が見つかりません: {', '.join(missing_columns)}")
            print(f"共通住民CSVの必須列: {', '.join(RESIDENT_COLUMNS)}（templates/residents.csv を参照）")
        elif len(addresses_df) == 0:
            print("データ行が1件もありません（見出し行だけのCSVです）。住民の行が入ったCSVを選択してください。")
        else:
            resident_id_problems = find_resident_id_problems(addresses_df)
            if resident_id_problems:
                print("resident_id の内容に問題があるため、処理を中止した。")
                for problem in resident_id_problems:
                    print(f"  - {problem}")
                print("resident_id を空欄・重複のない一意な値に修正してから、再度アップロードしてください。")
            else:
                batch_results = [geocode_one_address(addr) for addr in addresses_df["address"]]
                batch_df = pd.DataFrame(batch_results)

                # latitude/longitude/geocode_statusは常に最新の変換結果で置き換える
                # （再変換時、住所を修正した行の古い座標・状態が残らないようにするため）。
                addresses_df["latitude"] = batch_df["latitude"]
                addresses_df["longitude"] = batch_df["longitude"]
                addresses_df["geocode_status"] = batch_df["status"]

                # ABR内部確認用の列は、正式運用CSVには含めずNotebook上の確認表示にのみ使う。
                debug_df = addresses_df[["resident_id", "address", "geocode_status"]].copy()
                for col in DEBUG_COLUMNS:
                    debug_df[col] = batch_df[col]

                total_count = len(addresses_df)
                matched_count = int((addresses_df["geocode_status"] == "matched").sum())
                unmatched_count = total_count - matched_count

                print(f"全件数: {total_count}件")
                print(f"matched件数: {matched_count}件")
                print(f"未変換件数: {unmatched_count}件")

                print("\ngeocode_status内訳:")
                print(addresses_df["geocode_status"].value_counts().to_string())

                matched_mask = addresses_df["geocode_status"] == "matched"
                suffix_mask = matched_mask & (debug_df["ignored_suffix"].fillna("") != "")
                residential_count = int((matched_mask & (debug_df["address_type"] == "住居表示")).sum())
                last_number_mask = matched_mask & (debug_df["ignored_number"].fillna("") != "")
                print(f"\n住居表示住所として照合できた住所: {residential_count}件")
                print(f"方書を除外して照合できた住所: {int(suffix_mask.sum())}件")
                print(f"住居表示の末尾番号（3段目）を部屋番号等として除外して照合できた住所: {int(last_number_mask.sum())}件")

                print("\n先頭10件（確認用。normalized_address等のABR内部確認用列は出力CSVには含まれない）:")
                display(debug_df.head(10))

                if suffix_mask.any():
                    print("\n方書を除外して照合できた行（除外した部分 ignored_suffix が方書であることを確認する）:")
                    display(debug_df[suffix_mask])

                if last_number_mask.any():
                    print("\n住居表示の末尾番号を部屋番号等として除外して照合できた行（3段目の完全一致が無く、3段目を除いた"
                          "住居がABRに1件だけあった。除外した番号 ignored_number が部屋番号等であることを確認する）:")
                    display(debug_df.loc[last_number_mask, ["resident_id", "address", "abr_address", "ignored_number",
                                                            "ignored_suffix", "geocode_status"]])

                if unmatched_count:
                    unmatched_mask = addresses_df["geocode_status"] != "matched"
                    print(f"\nmatched以外の行（{unmatched_count}件。住所を確認・修正してこのセルを再実行できる）:")
                    display(debug_df[unmatched_mask])

                output_df = addresses_df[RESIDENT_COLUMNS]
                output_filename = build_geocoded_filename(csv_filename)
                output_df.to_csv(output_filename, index=False, encoding="utf-8-sig")
                print(
                    f"\n'{output_filename}' を出力した（UTF-8 BOM付き、"
                    "resident_id,address,latitude,longitude,geocode_status の5列）。"
                )
                files.download(output_filename)
